# 1. 라이브러리 Import

In [19]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print("=" * 60)
print("라이브러리 로드 완료!")
print("=" * 60)

라이브러리 로드 완료!


# 2. 데이터 로드

In [20]:
base_path = "data"
meta_path = os.path.join(base_path, "metadata.csv")
df = pd.read_csv(meta_path)
print(df.shape)
df.head()

(7565, 10)


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN


# 3. start_time 전처리

In [21]:
def clear_time(x):
    if pd.isna(x): return pd.NaT
    x = str(x)
    nums = []
    current = ''
    for ch in x:
        if ch.isdigit() or ch in ['.', 'e', 'E', '+', '-']:
            current += ch
        else:
            if current != '':
                nums.append(current)
                current = ''
    if current != '': nums.append(current)
    if len(nums) < 6: return pd.NaT
    try: nums = list(map(float, nums[:6]))
    except: return pd.NaT
    year, month, day, hour, minute, second = nums
    if not (2000 <= year <= 2100 and 1 <= month <= 12 and
            1 <= day <= 31 and 0 <= hour < 24 and
            0 <= minute < 60 and 0 <= second < 60): return pd.NaT
    try: return pd.Timestamp(int(year), int(month), int(day), int(hour), int(minute), int(second))
    except: return pd.NaT

df['start_time'] = df['start_time'].apply(clear_time)
df = df.dropna(subset=['start_time'])

### 3-1. 시간 파생 컬럼 생성

- `elapsed_hours_from_first`: 첫 가동 후 경과 시간
- `hours_from_prev_operation`: 직전 작업 시작시각과의 차이 (휴지기 + 작업시간 혼재)

In [22]:
df = df.sort_values(by=['battery_id', 'start_time']).reset_index(drop=True)

df['elapsed_hours_from_first'] = df.groupby('battery_id')['start_time'].transform(
    lambda x: (x - x.min()).dt.total_seconds() / 3600
)

# 변수명 변경 + 주의 주석
# 주의: 직전 discharge 이후 순수 휴지시간이 아니라
#       charge·discharge·impedance 포함한 직전 작업 시작시각과의 차이
df['hours_from_prev_operation'] = (
    df.groupby('battery_id')['start_time'].diff().dt.total_seconds() / 3600
)
df['hours_from_prev_operation'] = df['hours_from_prev_operation'].fillna(0)

# 4. 그룹 컬럼 추가

In [23]:
group_map = {
    'B0005':'A','B0006':'A','B0007':'A','B0018':'A',
    'B0025':'B','B0026':'B','B0027':'B','B0028':'B',
    'B0029':'C','B0030':'C','B0031':'C','B0032':'C',
    'B0033':'D','B0034':'D','B0036':'D',
    'B0038':'E','B0039':'E','B0040':'E',
    'B0041':'F','B0042':'F','B0043':'F','B0044':'F',
    'B0045':'G','B0046':'G','B0047':'G','B0048':'G',
    'B0049':'H','B0050':'H','B0051':'H','B0052':'H',
    'B0053':'I','B0054':'I','B0055':'I','B0056':'I',
}

end_reason_map = {
    'A': 'EOL',      'B': 'censored', 'C': 'censored',
    'D': 'QA_issue', 'E': 'QA_issue', 'F': 'QA_issue',
    'G': 'QA_issue', 'H': 'crashed',  'I': 'QA_issue',
}

analysis_role_map = {
    'A': 'main',    'B': 'excluded',  'C': 'comparison',
    'D': 'excluded','E': 'excluded',  'F': 'anomaly',
    'G': 'anomaly', 'H': 'anomaly',   'I': 'anomaly',
}

df['group']         = df['battery_id'].map(group_map)
df['end_reason']    = df['group'].map(end_reason_map)
df['analysis_role'] = df['group'].map(analysis_role_map)

print("\n그룹별 배터리 수 및 역할:")
summary = df.groupby(['group','end_reason','analysis_role'])['battery_id'].nunique().reset_index()
summary.columns = ['group','end_reason','analysis_role','배터리수']
print(summary.to_string(index=False))


그룹별 배터리 수 및 역할:
group end_reason analysis_role  배터리수
    A        EOL          main     4
    B   censored      excluded     4
    C   censored    comparison     4
    D   QA_issue      excluded     3
    E   QA_issue      excluded     3
    F   QA_issue       anomaly     4
    G   QA_issue       anomaly     4
    H    crashed       anomaly     4
    I   QA_issue       anomaly     4


# 5. 실험 조건 라벨 & battery_protocol_map

- `test_temperature_profile`: 원본 `ambient_temperature`와 분리된 해석용 라벨
-  F그룹 `4°C_22°C_mixed`
- `battery_protocol_map`: cutoff_voltage battery_id 기준 관리

In [24]:
#ambient_temperature: 원본값 유지
#test_temperature_profile: 해석용 라벨 (분리)
test_temperature_profile_map = {
    'A': '24°C_stable',    'B': '24°C_stable',
    'C': '43°C_stable',    'D': '24°C_stable',
    'E': '24_44°C_mixed',
    'F': '4°C_22°C_mixed',  # 실제 데이터에 4°C와 22°C 혼재
    'G': '4°C_stable',    'H': '4°C_stable',    'I': '4°C_stable',
}

load_profile_map = {
    'A': '2A_CC',         'B': '4A_squarewave', 'C': '4A_CC',
    'D': '2A_4A_mixed',   'E': '1A_2A_4A_mixed','F': '4A_1A_mixed',
    'G': '1A_CC',         'H': '2A_CC',         'I': '2A_CC',
}

eol_rule_source_map = {
    'A': 'NASA_30%fade', 'B': 'censored',      'C': 'censored',
    'D': 'NASA_20%fade', 'E': 'NASA_20%fade',
    'F': 'NASA_30%fade', 'G': 'NASA_30%fade',
    'H': 'crashed',      'I': 'NASA_30%fade',
}

df['test_temperature_profile'] = df['group'].map(test_temperature_profile_map)
df['load_profile']             = df['group'].map(load_profile_map)
df['eol_rule_source']          = df['group'].map(eol_rule_source_map)

In [25]:
#  battery_protocol_map
battery_protocol_map = {
    'B0005': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0006': {'cutoff_voltage': 2.5, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0007': {'cutoff_voltage': 2.2, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0018': {'cutoff_voltage': 2.5, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0025': {'cutoff_voltage': 2.7, 'discharge_current': '4A_squarewave', 'eol_fade': 0.30},
    'B0026': {'cutoff_voltage': 2.7, 'discharge_current': '4A_squarewave', 'eol_fade': 0.30},
    'B0027': {'cutoff_voltage': 2.7, 'discharge_current': '4A_squarewave', 'eol_fade': 0.30},
    'B0028': {'cutoff_voltage': 2.7, 'discharge_current': '4A_squarewave', 'eol_fade': 0.30},
    'B0029': {'cutoff_voltage': 2.7, 'discharge_current': '4A',            'eol_fade': 0.30},
    'B0030': {'cutoff_voltage': 2.7, 'discharge_current': '4A',            'eol_fade': 0.30},
    'B0031': {'cutoff_voltage': 2.7, 'discharge_current': '4A',            'eol_fade': 0.30},
    'B0032': {'cutoff_voltage': 2.7, 'discharge_current': '4A',            'eol_fade': 0.30},
    'B0033': {'cutoff_voltage': 2.0, 'discharge_current': '4A_2A_mixed',   'eol_fade': 0.20},
    'B0034': {'cutoff_voltage': 2.0, 'discharge_current': '4A_2A_mixed',   'eol_fade': 0.20},
    'B0036': {'cutoff_voltage': 2.0, 'discharge_current': '4A_2A_mixed',   'eol_fade': 0.20},
    'B0038': {'cutoff_voltage': 2.2, 'discharge_current': '1A_2A_4A_mixed','eol_fade': 0.20},
    'B0039': {'cutoff_voltage': 2.2, 'discharge_current': '1A_2A_4A_mixed','eol_fade': 0.20},
    'B0040': {'cutoff_voltage': 2.2, 'discharge_current': '1A_2A_4A_mixed','eol_fade': 0.20},
    'B0041': {'cutoff_voltage': 2.7, 'discharge_current': '4A_1A_mixed',   'eol_fade': 0.30},
    'B0042': {'cutoff_voltage': 2.7, 'discharge_current': '4A_1A_mixed',   'eol_fade': 0.30},
    'B0043': {'cutoff_voltage': 2.7, 'discharge_current': '4A_1A_mixed',   'eol_fade': 0.30},
    'B0044': {'cutoff_voltage': 2.7, 'discharge_current': '4A_1A_mixed',   'eol_fade': 0.30},
    'B0045': {'cutoff_voltage': 2.7, 'discharge_current': '1A',            'eol_fade': 0.30},
    'B0046': {'cutoff_voltage': 2.7, 'discharge_current': '1A',            'eol_fade': 0.30},
    'B0047': {'cutoff_voltage': 2.7, 'discharge_current': '1A',            'eol_fade': 0.30},
    'B0048': {'cutoff_voltage': 2.7, 'discharge_current': '1A',            'eol_fade': 0.30},
    'B0049': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': None},
    'B0050': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': None},
    'B0051': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': None},
    'B0052': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': None},
    'B0053': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0054': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0055': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': 0.30},
    'B0056': {'cutoff_voltage': 2.7, 'discharge_current': '2A',            'eol_fade': 0.30},
}

df['cutoff_voltage']     = df['battery_id'].map({k: v['cutoff_voltage']    for k, v in battery_protocol_map.items()})
df['discharge_current']  = df['battery_id'].map({k: v['discharge_current'] for k, v in battery_protocol_map.items()})
df['eol_fade_threshold'] = df['battery_id'].map({k: v['eol_fade']          for k, v in battery_protocol_map.items()})

print("=" * 55)
print("battery_protocol_map 적용 완료")
print("=" * 55)
print(df.groupby('battery_id')[['group','cutoff_voltage','discharge_current']].first().to_string())

battery_protocol_map 적용 완료
           group  cutoff_voltage discharge_current
battery_id                                        
B0005          A             2.7                2A
B0006          A             2.5                2A
B0007          A             2.2                2A
B0018          A             2.5                2A
B0025          B             2.7     4A_squarewave
B0026          B             2.7     4A_squarewave
B0027          B             2.7     4A_squarewave
B0028          B             2.7     4A_squarewave
B0029          C             2.7                4A
B0030          C             2.7                4A
B0031          C             2.7                4A
B0032          C             2.7                4A
B0033          D             2.0       4A_2A_mixed
B0034          D             2.0       4A_2A_mixed
B0036          D             2.0       4A_2A_mixed
B0038          E             2.2    1A_2A_4A_mixed
B0039          E             2.2    1A_2A_4A_mixed
B004

# 6. 타입별 3개 테이블 분리

In [26]:
df_charge    = df[df['type'] == 'charge'].copy()
df_discharge = df[df['type'] == 'discharge'].copy()
df_impedance = df[df['type'] == 'impedance'].copy()

print("\n" + "=" * 55)
print("타입별 분리 완료")
print("=" * 55)
print(f"df_charge    : {len(df_charge):,}행")
print(f"df_discharge : {len(df_discharge):,}행")
print(f"df_impedance : {len(df_impedance):,}행")


타입별 분리 완료
df_charge    : 2,815행
df_discharge : 2,794행
df_impedance : 1,956행


# 7. Capacity 상태 flag 추가

In [27]:
df_discharge['Capacity'] = pd.to_numeric(df_discharge['Capacity'], errors='coerce')

def classify_capacity(cap):
    if pd.isna(cap):   return 'missing'
    elif cap == 0:     return 'zero'
    elif cap > 2.1:    return 'impossible_high'
    elif cap < 0.3:    return 'low_anomaly'
    else:              return 'valid'

df_discharge['cap_flag']   = df_discharge['Capacity'].apply(classify_capacity)
hard_exclude = ['missing', 'zero', 'impossible_high']
df_discharge['cap_exclude'] = df_discharge['cap_flag'].isin(hard_exclude)
df_discharge['cap_anomaly'] = df_discharge['cap_flag'] == 'low_anomaly'

print("\n" + "=" * 55)
print("Capacity flag 분포")
print("=" * 55)
print(df_discharge['cap_flag'].value_counts())
print("\n하드 제외 후보:", df_discharge['cap_exclude'].sum())
print("이상탐지 대상:", df_discharge['cap_anomaly'].sum())


Capacity flag 분포
cap_flag
valid              2553
low_anomaly         193
missing              25
zero                 19
impossible_high       4
Name: count, dtype: int64

하드 제외 후보: 48
이상탐지 대상: 193


### 7-1. Impedance 상태 flag 추가

In [28]:
df_impedance['Re']  = pd.to_numeric(df_impedance['Re'],  errors='coerce')
df_impedance['Rct'] = pd.to_numeric(df_impedance['Rct'], errors='coerce')
df_impedance = df_impedance.sort_values(['battery_id', 'start_time']).reset_index(drop=True)
df_impedance['re_diff'] = df_impedance.groupby('battery_id')['Re'].diff().abs()

def classify_impedance(row):
    re, rct, diff = row['Re'], row['Rct'], row['re_diff']
    if pd.isna(re) or pd.isna(rct):   return 'missing'
    if re <= 0 or rct <= 0:           return 'zero_or_minus'
    if pd.notna(diff) and diff > 0.3: return 'noise_candidate'
    if rct > re * 10:                 return 'rct_imbalance'
    if re > 1.0 or rct > 1.5:        return 'high_anomaly'
    return 'valid'

df_impedance['imp_flag'] = df_impedance.apply(classify_impedance, axis=1)
hard_exclude_imp = ['missing', 'zero_or_minus']
df_impedance['imp_exclude'] = df_impedance['imp_flag'].isin(hard_exclude_imp)
anomaly_imp = ['high_anomaly', 'noise_candidate', 'rct_imbalance']
df_impedance['imp_anomaly'] = df_impedance['imp_flag'].isin(anomaly_imp)

print("=" * 55)
print("Impedance Flag 분포")
print("=" * 55)
print(df_impedance['imp_flag'].value_counts())
print(f"\n하드 제외: {df_impedance['imp_exclude'].sum()}건")
print(f"이상탐지: {df_impedance['imp_anomaly'].sum()}건")

Impedance Flag 분포
imp_flag
valid              1932
zero_or_minus        11
missing               9
noise_candidate       4
Name: count, dtype: int64

하드 제외: 20건
이상탐지: 4건


# 8. discharge 사이클 순번

In [29]:
df_discharge = df_discharge.sort_values(
    ['battery_id', 'start_time', 'filename']
).reset_index(drop=True)

df_discharge['discharge_cycle_raw'] = (
    df_discharge.groupby('battery_id').cumcount() + 1
)

df_discharge['is_hard_excluded'] = df_discharge['cap_flag'].isin(
    ['missing', 'zero', 'impossible_high']
)

df_discharge['discharge_cycle_valid'] = (
    df_discharge.loc[~df_discharge['is_hard_excluded']]
    .groupby('battery_id').cumcount() + 1
)

# 9. 초기 Capacity 계산

In [30]:
init_cap = (
    df_discharge[df_discharge['cap_flag'] == 'valid']
    .groupby('battery_id')['Capacity']
    .apply(lambda x: x.head(5).median())
    .rename('init_cap')
)
df_discharge = df_discharge.merge(init_cap, on='battery_id', how='left')

print("\n초기 Capacity (valid 첫 5개 중앙값):")
print(df_discharge.groupby(['battery_id','group'])['init_cap'].first().reset_index().round(4).to_string(index=False))


초기 Capacity (valid 첫 5개 중앙값):
battery_id group  init_cap
     B0005     A    1.8353
     B0006     A    2.0133
     B0007     A    1.8807
     B0018     A    1.8396
     B0025     B    1.8471
     B0026     B    1.8143
     B0027     B    1.8142
     B0028     B    1.7976
     B0029     C    1.8158
     B0030     C    1.7518
     B0031     C    1.8044
     B0032     C    1.8655
     B0033     D    1.2529
     B0034     D    1.6207
     B0036     D    1.8011
     B0038     E    1.0613
     B0039     E    0.4711
     B0040     E    0.7796
     B0041     F    1.1195
     B0042     F    1.7282
     B0043     F    1.6815
     B0044     F    1.6534
     B0045     G    0.8852
     B0046     G    1.5031
     B0047     G    1.5081
     B0048     G    1.4989
     B0049     H    1.3644
     B0050     H    1.5518
     B0051     H    1.2039
     B0052     H    1.3611
     B0053     I    1.1306
     B0054     I    1.0960
     B0055     I    1.2573
     B0056     I    1.2974


# 10. SOH 계산

In [31]:
df_discharge['SOH_nominal'] = np.where(
    df_discharge['cap_flag'] == 'valid',
    (df_discharge['Capacity'] / 2.0 * 100).round(2), np.nan
)

# SOH_relative 100% 초과 주석
# 이유: init_cap = valid 첫 5개 중앙값 기준이라
#       초반 측정값이 중앙값보다 살짝 높으면 100% 초과 가능
# → 계산 오류 아님. 정상적인 구조적 특성.
df_discharge['SOH_relative'] = np.where(
    df_discharge['cap_flag'] == 'valid',
    (df_discharge['Capacity'] / df_discharge['init_cap'] * 100).round(2), np.nan
)

# 11. EOL 탐지 & RUL 계산

 `rul_label_type` 추가 — main만 supervised

In [32]:
for col in ['eol_cycle', 'RUL', 'eol_soh_threshold']:
    if col in df_discharge.columns:
        df_discharge = df_discharge.drop(columns=col)

eol_threshold_map = {
    'A': 70, 'B': 70, 'C': 70,
    'D': 80, 'E': 80,
    'F': 70, 'G': 70, 'H': None, 'I': 70
}
df_discharge['eol_soh_threshold'] = df_discharge['group'].map(eol_threshold_map)

eol_cycles = (
    df_discharge[
        (df_discharge['cap_flag'] == 'valid') &
        (df_discharge['eol_soh_threshold'].notna()) &
        (df_discharge['SOH_nominal'] < df_discharge['eol_soh_threshold'])
    ]
    .groupby('battery_id')['discharge_cycle_raw'].min()
    .rename('eol_cycle').reset_index()
)
df_discharge = df_discharge.merge(eol_cycles, on='battery_id', how='left')

df_discharge['RUL'] = np.where(
    df_discharge['eol_cycle'].notna(),
    (df_discharge['eol_cycle'] - df_discharge['discharge_cycle_raw']).clip(lower=0),
    np.nan
)

# [수정 6] rul_label_type
df_discharge['rul_label_type'] = np.where(
    df_discharge['analysis_role'] == 'main',       'supervised',
np.where(
    df_discharge['analysis_role'] == 'comparison', 'censored',
np.where(
    df_discharge['analysis_role'] == 'anomaly',    'anomaly_case',
    'unsupported_for_rul'
)))

print("\nrul_label_type 분포:")
print(df_discharge['rul_label_type'].value_counts())

eol_summary = (
    df_discharge.groupby('battery_id')
    .agg(
        group=('group','first'), analysis_role=('analysis_role','first'),
        rul_label_type=('rul_label_type','first'),
        eol_cycle=('eol_cycle','first'), total_cycles=('discharge_cycle_raw','max'),
        rul_available=('RUL', lambda x: x.notna().sum()),
    ).reset_index()
)
eol_summary['eol_달성'] = eol_summary['eol_cycle'].notna().map({True:'O',False:'X'})
print("\n" + "=" * 55)
print("전체 그룹 EOL & RUL 요약")
print("=" * 55)
print(eol_summary.to_string(index=False))


rul_label_type 분포:
rul_label_type
anomaly_case           1154
unsupported_for_rul     844
supervised              636
censored                160
Name: count, dtype: int64

전체 그룹 EOL & RUL 요약
battery_id group analysis_role      rul_label_type  eol_cycle  total_cycles  rul_available eol_달성
     B0005     A          main          supervised      125.0           168            168      O
     B0006     A          main          supervised      109.0           168            168      O
     B0007     A          main          supervised        NaN           168              0      X
     B0018     A          main          supervised       97.0           132            132      O
     B0025     B      excluded unsupported_for_rul        NaN            28              0      X
     B0026     B      excluded unsupported_for_rul        6.0            28             28      O
     B0027     B      excluded unsupported_for_rul        NaN            28              0      X
     B0028     B      e

# 12. 그룹별 DataFrame 분리

In [33]:
df_main       = df_discharge[df_discharge['analysis_role'] == 'main'].copy()
df_comparison = df_discharge[df_discharge['analysis_role'] == 'comparison'].copy()
df_anomaly    = df_discharge[df_discharge['analysis_role'] == 'anomaly'].copy()
df_excluded   = df_discharge[df_discharge['analysis_role'] == 'excluded'].copy()

# 13. impedance 전처리 및 역할별 분리

In [34]:
df_impedance['group']         = df_impedance['battery_id'].map(group_map)
df_impedance['end_reason']    = df_impedance['group'].map(end_reason_map)
df_impedance['analysis_role'] = df_impedance['group'].map(analysis_role_map)
df_impedance = df_impedance.sort_values(['battery_id','test_id'])
df_impedance['impedance_cycle_no'] = df_impedance.groupby('battery_id').cumcount() + 1

df_imp_main       = df_impedance[df_impedance['analysis_role'] == 'main'].copy()
df_imp_comparison = df_impedance[df_impedance['analysis_role'] == 'comparison'].copy()
df_imp_anomaly    = df_impedance[df_impedance['analysis_role'] == 'anomaly'].copy()

group_dfs = {grp: df_discharge[df_discharge['group']==grp].copy()
             for grp in sorted(df_discharge['group'].unique())}

print(f"df_imp_main       : {len(df_imp_main):>4}행")
print(f"df_imp_comparison : {len(df_imp_comparison):>4}행")
print(f"df_imp_anomaly    : {len(df_imp_anomaly):>4}행")
print(f"\ndf_main       : {len(df_main):>4}행  {df_main['battery_id'].nunique()}개 배터리")
print(f"df_comparison : {len(df_comparison):>4}행  {df_comparison['battery_id'].nunique()}개 배터리")
print(f"df_anomaly    : {len(df_anomaly):>4}행  {df_anomaly['battery_id'].nunique()}개 배터리")

df_imp_main       :  887행
df_imp_comparison :   68행
df_imp_anomaly    :  557행

df_main       :  636행  4개 배터리
df_comparison :  160행  4개 배터리
df_anomaly    : 1154행  16개 배터리


# 14. ML용 데이터셋 생성


- `merge_asof` backward join: cycle 근사 매핑 제거 → start_time 기준 과거값만 사용
- `impedance_available` 플래그: 첫 impedance 이전/이후 구간 구분
-  조건부 ffill: 이전 구간 NaN 유지, 이후 구간만 ffill
- `battery_id` feature 제외 (GroupKFold용으로만 유지)

In [35]:
# ============================================================
# merge_asof backward join
# ============================================================
# 기존 문제: impedance_cycle_no // 2 근사 매핑
#   → B0005·B0006·B0007: discharge 168개, impedance 278개 
#   → B0018: discharge 132개, impedance 53개 
#   → 매핑 불일치로 163개 비정상 결측 발생
#
# 수정: start_time 기준 backward asof join
#   → 현재 discharge 시점보다 이전에 측정된 가장 최근 impedance 사용
#   → 미래 impedance 정보 누수 완전 차단
#   → 정상적인 결측만 남음 (첫 impedance 이전 구간)
# ============================================================

imp_A = df_imp_main[['battery_id','start_time','Re','Rct']].copy()
imp_A['Re']  = pd.to_numeric(imp_A['Re'],  errors='coerce')
imp_A['Rct'] = pd.to_numeric(imp_A['Rct'], errors='coerce')
imp_A = imp_A.sort_values(['battery_id','start_time']).reset_index(drop=True)

# cumulative mean 계산 (현재까지 관측된 누적 평균)
imp_A['Re_cumean']  = (
    imp_A.groupby('battery_id')['Re']
    .expanding().mean().reset_index(level=0, drop=True)
)
imp_A['Rct_cumean'] = (
    imp_A.groupby('battery_id')['Rct']
    .expanding().mean().reset_index(level=0, drop=True)
)

# discharge 데이터 준비
dis_A = (
    df_main[df_main['cap_flag'] == 'valid']
    [['battery_id','start_time','discharge_cycle_raw',
      'discharge_cycle_valid','SOH_relative','SOH_nominal',
      'ambient_temperature','RUL','rul_label_type']]
    .dropna(subset=['RUL'])
    .query("rul_label_type == 'supervised'")
    .sort_values(['battery_id','start_time'])
    .reset_index(drop=True)
    .copy()
)

# 배터리별 merge_asof backward join
results = []
for bid in dis_A['battery_id'].unique():
    d = dis_A[dis_A['battery_id'] == bid].sort_values('start_time').reset_index(drop=True)
    i = imp_A[imp_A['battery_id'] == bid][['start_time','Re_cumean','Rct_cumean']].sort_values('start_time').reset_index(drop=True)
    merged = pd.merge_asof(d, i, on='start_time', direction='backward')
    results.append(merged)

df_ml = pd.concat(results, ignore_index=True)
df_ml = df_ml.rename(columns={'Re_cumean': 'Re_mean', 'Rct_cumean': 'Rct_mean'})

# impedance_available 플래그
# 첫 impedance 측정 시점 기준으로 이전/이후 구간 구분
first_imp = (
    imp_A.groupby('battery_id')['start_time']
    .min().rename('first_imp_time')
)
df_ml = df_ml.merge(first_imp, on='battery_id', how='left')
df_ml['impedance_available'] = df_ml['start_time'] >= df_ml['first_imp_time']

# 조건부 ffill
# 결측 두 종류로 분리:
#  1. 첫 impedance 이전 구간 → 과거 정보 없음 → NaN 유지 (정상 결측)
#  2. 첫 impedance 이후 구간 → 마지막 cumulative mean 유지 → ffill
df_ml = df_ml.sort_values(['battery_id', 'discharge_cycle_raw'])
df_ml['Re_mean']  = df_ml.groupby('battery_id')['Re_mean'].ffill()
df_ml['Rct_mean'] = df_ml.groupby('battery_id')['Rct_mean'].ffill()
# ffill 후에도 NaN이면 → 첫 impedance 이전 구간 (정상 결측, NaN 유지)

print("=" * 55)
print("ML용 데이터셋")
print("=" * 55)
print(f"총 행 수                : {len(df_ml)}")
print(f"배터리 수               : {df_ml['battery_id'].nunique()}")
print(f"impedance 이전 구간(NaN): {(~df_ml['impedance_available']).sum()}행 (정상 결측)")
print(f"impedance 이후 구간    : {df_ml['impedance_available'].sum()}행")
print(f"Re_mean NaN (정상 결측): {df_ml['Re_mean'].isna().sum()}")
print(f"Rct_mean NaN (정상 결측): {df_ml['Rct_mean'].isna().sum()}")
print(f"\n결측치 전체:")
print(df_ml.isnull().sum())
print(f"\n샘플 (앞 5행):")
print(df_ml[['battery_id','discharge_cycle_raw','SOH_relative','Re_mean','Rct_mean','RUL','impedance_available']].head(10).to_string(index=False))

ML용 데이터셋
총 행 수                : 468
배터리 수               : 3
impedance 이전 구간(NaN): 38행 (정상 결측)
impedance 이후 구간    : 430행
Re_mean NaN (정상 결측): 38
Rct_mean NaN (정상 결측): 38

결측치 전체:
battery_id                0
start_time                0
discharge_cycle_raw       0
discharge_cycle_valid     0
SOH_relative              0
SOH_nominal               0
ambient_temperature       0
RUL                       0
rul_label_type            0
Re_mean                  38
Rct_mean                 38
first_imp_time            0
impedance_available       0
dtype: int64

샘플 (앞 5행):
battery_id  discharge_cycle_raw  SOH_relative  Re_mean  Rct_mean   RUL  impedance_available
     B0005                    1        101.15      NaN       NaN 124.0                False
     B0005                    2        100.60      NaN       NaN 123.0                False
     B0005                    3        100.00      NaN       NaN 122.0                False
     B0005                    4        100.00      NaN       NaN 

# 15. 전체 저장

In [36]:
os.makedirs(base_path, exist_ok=True)

df_discharge.to_csv(os.path.join(base_path, 'df_discharge_processed.csv'), index=False)
df_main.to_csv(os.path.join(base_path, 'df_A_main.csv'), index=False)
df_comparison.to_csv(os.path.join(base_path, 'df_C_comparison.csv'), index=False)
df_anomaly.to_csv(os.path.join(base_path, 'df_anomaly.csv'), index=False)

for grp, gdf in group_dfs.items():
    gdf.to_csv(os.path.join(base_path, f'df_group_{grp}.csv'), index=False)

df_impedance.to_csv(os.path.join(base_path, 'df_impedance_processed.csv'), index=False)
df_imp_main.to_csv(os.path.join(base_path, 'df_imp_A_main.csv'), index=False)
df_imp_comparison.to_csv(os.path.join(base_path, 'df_imp_C_comparison.csv'), index=False)

df_ml.to_csv(os.path.join(base_path, 'df_ml_dataset.csv'), index=False)

print("\n" + "=" * 55)
print("전체 저장 완료")
print("=" * 55)
print("df_discharge_processed.csv  — 전체 34개 배터리 discharge")
print("df_A_main.csv               — 그룹A 메인 분석")
print("df_C_comparison.csv         — 그룹C 온도 비교")
print("df_anomaly.csv              — 그룹F·G·H·I 이상탐지")
print("df_group_{A~I}.csv          — 그룹별 개별 파일")
print("df_impedance_processed.csv  — 전체 impedance")
print("df_imp_A_main.csv           — 그룹A impedance")
print("df_imp_C_comparison.csv     — 그룹C impedance")
print("df_ml_dataset.csv           — ML 학습용 데이터셋")
print("\n다음 단계: 02_EDA.ipynb")


전체 저장 완료
df_discharge_processed.csv  — 전체 34개 배터리 discharge
df_A_main.csv               — 그룹A 메인 분석
df_C_comparison.csv         — 그룹C 온도 비교
df_anomaly.csv              — 그룹F·G·H·I 이상탐지
df_group_{A~I}.csv          — 그룹별 개별 파일
df_impedance_processed.csv  — 전체 impedance
df_imp_A_main.csv           — 그룹A impedance
df_imp_C_comparison.csv     — 그룹C impedance
df_ml_dataset.csv           — ML 학습용 데이터셋

다음 단계: 02_EDA.ipynb
